# Week 8 (Tue) - Class Activities in CS 315

You are given the file `netflix_top10_countries.tsv` in the folder data, solve the following tasks using it.

**Table of Contents**

1. [Q1: Load the file](#q1)
2. [Q2: Show number of rows](#q2)
3. [Q3: Show column names](#q3)
4. [Q4: Show number of total countries](#q4)
5. [Q5: Show number of TV series](#q5)
6. [Q6: Number of TV series in South Korea](#q6)
7. [Q7: Total weeks of Squid Games in Italy ](#q7)
8. [Q8: Time frame for a given country](#q8)
9. [Q9: Is number of weeks in ranking equal?](#q9)
10. [Q10: Country with fewest weeks](#q10)


<a id="q1"></a>
### Q1: Load the file

In [11]:
import pandas as pd

df = pd.read_csv('/Users/wuyuwei/Desktop/all-weeks-countries.tsv', sep='\t')
df.head()

,country_name,country_iso2,week,category,weekly_rank,show_title,season_title,cumulative_weeks_in_top_10
0,Argentina,AR,2026-10-18,Films,1,Therapy Done Right,NaN,1
1,Argentina,AR,2026-10-18,Films,2,Animals,NaN,2
2,Argentina,AR,2026-10-18,Films,3,Anyone But You,NaN,2
3,Argentina,AR,2026-10-18,Films,4,Doing Life,NaN,3
4,Argentina,AR,2026-10-18,Films,5,Ice Age: Collision Course,NaN,5


<a id="q2"></a>
### Q2: Show number of rows

In [12]:
len(df)

515920

<a id="q3"></a>
### Q3: Show column names

In [13]:
df.columns.tolist()

['country_name',
 'country_iso2',
 'week',
 'category',
 'weekly_rank',
 'show_title',
 'season_title',
 'cumulative_weeks_in_top_10']

<a id="q4"></a>
### Q4: Show number of total countries

In [14]:
df['country_name'].nunique()

94

<a id="q5"></a>
### Q5: Show number of TV series

In [15]:
df[df['category'].str.contains('TV')]['show_title'].nunique()

3607

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [16]:
sk_tv = df[(df['country_name'] == 'South Korea') & (df['category'].str.contains('TV'))]
sk_tv['show_title'].nunique()

486

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [17]:
squid_italy = df[(df['country_name'] == 'Italy') & (df['show_title'].str.contains('Squid Game', case=False, na=False))]
squid_italy['cumulative_weeks_in_top_10'].max()

np.int64(21)

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [18]:
def get_timeframe(country_name):
    country_df = df[df['country_name'] == country_name]
    start_date = country_df['week'].min()
    end_date = country_df['week'].max()
    print(f"{country_name}: {start_date} to {end_date}")

get_timeframe('South Korea')
get_timeframe('United States')
get_timeframe('Argentina')
get_timeframe('Russia')

South Korea: 2021-07-04 to 2026-10-18
United States: 2021-07-04 to 2026-10-18
Argentina: 2021-07-04 to 2026-10-18
Russia: 2021-07-04 to 2022-02-27


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

The total number of weeks in the rankings is `len(df['week'].unique())`.

The total number of countries is `len(df['country_name'].unique())`.  

For every week, each country has 20 rows.   

If they all stayed the same number of weeks, then the product of these three weeks would be the same as the total number of rows.  

In [19]:
num_weeks = len(df['week'].unique())
num_countries = len(df['country_name'].unique())
expected_rows = num_weeks * num_countries * 20

len(df) == expected_rows

False

We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Write a one-liner that will show the country with the fewest weeks in the Netflix rankings

In [20]:
df.groupby('country_name')['week'].nunique().idxmin()

'Russia'

In [21]:
def rank2score(rank):
    f1_points = {1: 25, 2: 18, 3: 15, 4: 12, 5: 10, 6: 8, 7: 6, 8: 4, 9: 2, 10: 1}
    return f1_points.get(rank, 0)

df['score'] = df['weekly_rank'].apply(rank2score)
df[['country_name', 'week', 'weekly_rank', 'show_title', 'score', 'cumulative_weeks_in_top_10']].head(10)

,country_name,week,weekly_rank,show_title,score,cumulative_weeks_in_top_10
0,Argentina,2026-10-18,1,Therapy Done Right,25,1
1,Argentina,2026-10-18,2,Animals,18,2
2,Argentina,2026-10-18,3,Anyone But You,15,2
3,Argentina,2026-10-18,4,Doing Life,12,3
4,Argentina,2026-10-18,5,Ice Age: Collision Course,10,5
5,Argentina,2026-10-18,6,Corpse Bride,8,1
6,Argentina,2026-10-18,7,Acid,6,2
7,Argentina,2026-10-18,8,Ice Age: Dawn of the Dinosaurs,4,6
8,Argentina,2026-10-18,9,The Storm,2,1
9,Argentina,2026-10-18,10,The Equalizer 3,1,3


In [22]:
target_show = "Lovely Runner"
show_df = df[df['show_title'] == target_show].sort_values('week')
results = []

for country, group in show_df.groupby('country_name'):
    total_score = group['score'].sum()
    total_weeks = len(group)
    rankings_list = group['weekly_rank'].tolist()
    results.append({
        'country_name': country,
        'total_score': total_score,
        'total_weeks': total_weeks,
        'rankings_list': rankings_list
    })

summary_df = pd.DataFrame(results).sort_values(by=['total_score', 'total_weeks'], ascending=False)
summary_df

,country_name,total_score,total_weeks,rankings_list
12,Vietnam,75,7,"[5, 1, 4, 5, 6, 7, 8]"
2,Indonesia,69,7,"[5, 1, 4, 7, 6, 7, 9]"
3,Malaysia,62,6,"[4, 1, 3, 7, 9, 9]"
6,Philippines,41,4,"[5, 3, 5, 7]"
4,Maldives,32,4,"[6, 3, 6, 10]"
9,Sri Lanka,32,4,"[5, 3, 7, 10]"
0,Bangladesh,30,3,"[6, 4, 5]"
1,India,26,3,"[5, 4, 8]"
7,Singapore,20,4,"[9, 5, 7, 9]"
11,Thailand,20,3,"[5, 8, 7]"
